In [ ]:
import torch
import numpy as np
import os
import nibabel as nib
from typing import Sequence, Tuple, Union

# -----------------------
# Utility: repeat value into tuple
# -----------------------
def ensure_tuple_rep(val: Union[float, Sequence[float]], length: int):
    if isinstance(val, (list, tuple)):
        if len(val) != length:
            raise ValueError(f"Expected tuple of length {length}, got {len(val)}.")
        return tuple(val)
    return tuple([val] * length)

# -----------------------
# Compute MONAI-like importance map
# -----------------------
def compute_importance_map_monai_like(
    patch_size: Tuple[int, ...],
    mode: str = "gaussian",
    sigma_scale: Union[Sequence[float], float] = 0.125,
    device: Union[str, torch.device] = "cpu",
    dtype: torch.dtype = torch.float32,
) -> torch.Tensor:
    device = torch.device(device)
    if mode.lower() == "constant":
        importance_map = torch.ones(patch_size, device=device, dtype=torch.float)
    elif mode.lower() == "gaussian":
        sigma_scale = ensure_tuple_rep(sigma_scale, len(patch_size))
        sigmas = [dim * sigma_s for dim, sigma_s in zip(patch_size, sigma_scale)]
        importance_map = None
        for i in range(len(patch_size)):
            coords = torch.arange(
                start=-(patch_size[i] - 1) / 2.0,
                end=(patch_size[i] - 1) / 2.0 + 1,
                dtype=torch.float,
                device=device,
            )
            gauss_1d = torch.exp(coords**2 / (-2 * sigmas[i] ** 2))
            if i > 0:
                importance_map = importance_map.unsqueeze(-1) * gauss_1d[(None,) * i]
            else:
                importance_map = gauss_1d
    else:
        raise ValueError("Unsupported mode")

    # Clamp min value like MONAI does
    min_non_zero = max(torch.min(importance_map).item(), 1e-3)
    importance_map = torch.clamp_(importance_map.to(torch.float), min=min_non_zero).to(dtype)
    return importance_map

# -----------------------
# Main: generate and save as NIfTI
# -----------------------
patch_size = (64, 64)  # Example 2D ROI size
sigma_values = [0.0, 0.01, 0.05, 0.125, 0.23, 0.5]  # Test different sigma_scale values

save_dir = "importance_maps"
os.makedirs(save_dir, exist_ok=True)

for s in sigma_values:
    try:
        imp_map = compute_importance_map_monai_like(patch_size, mode="gaussian", sigma_scale=s)
        arr = imp_map.cpu().numpy().astype(np.float32)

        # Add singleton z-dimension so it's valid 3D
        arr_3d = arr[:, :, np.newaxis]  # shape (H, W, 1)

        # Create NIfTI (identity affine)
        affine = np.eye(4)
        nifti_img = nib.Nifti1Image(arr_3d, affine)
        file_path = os.path.join(save_dir, f"importance_map_sigma_{s}.nii.gz")
        nib.save(nifti_img, file_path)

        print(f"Saved sigma_scale={s} map to {file_path}")
    except Exception as e:
        print(f"Error with sigma_scale={s}: {e}")


In [ ]:
from monai.data.utils import dense_patch_slices
from monai.inferers.utils import _get_scan_interval
def count_sliding_windows(image_size, roi_size, overlap=0.25):
    """
    Count how many patches (inferences) SlidingWindowInferer would run.

    Args:
      image_size (Sequence[int]): spatial shape of your input (e.g. [D, H, W] or [H, W]).
      roi_size    (Sequence[int]): the patch size you want to use.
      overlap     (float or Sequence[float]): fraction(s) of overlap between 0 and 1.

    Returns:
      int: the exact number of windows (= inferences) that will be executed.
    """
    num_dims = len(image_size)
    # ensure overlap is a sequence of length num_dims
    if isinstance(overlap, (float, int)):
        overlap = [float(overlap)] * num_dims
    elif len(overlap) != num_dims:
        raise ValueError(f"overlap must be a float or a sequence of length {num_dims}")

    # 1. Compute scan intervals (the “stride” per axis)
    scan_interval = _get_scan_interval(
        image_size=image_size,
        roi_size=roi_size,
        num_spatial_dims=num_dims,
        overlap=overlap,
    )
    # 2. Enumerate all patch‐slice definitions
    all_slices = dense_patch_slices(image_size, roi_size, scan_interval)
    # 3. Return how many there are
    return len(all_slices)

def compute_overlap(image_size, roi_size, target_num):
    """
    Compute the smallest scalar overlap such that the number of sliding windows is >= target_num.

    Uses binary search to find the smallest overlap where count_sliding_windows >= target_num.

    Args:
      image_size (Sequence[int]): spatial shape of your input (e.g. [D, H, W] or [H, W]).
      roi_size    (Sequence[int]): the patch size you want to use.
      target_num  (int): the desired number of windows.

    Returns:
      float: the smallest overlap value where count >= target_num, or None if impossible.
    """
    min_count = count_sliding_windows(image_size, roi_size, overlap=0.5)
    max_count = count_sliding_windows(image_size, roi_size, overlap=0.99)
    if target_num < min_count:
        return 0.5  
    if target_num > max_count:
        return overlap_here 

    low = 0.51
    high = 0.99
    best_overlap = high
    best_count = max_count

    for overlap_here in range(int(low*100), int(high*100), int(0.01*100)):
        overlap_here = overlap_here/100
        n_windows = count_sliding_windows(image_size, roi_size, overlap=overlap_here)
        if n_windows >= target_num:
            return best_overlap
        else:
            best_overlap = overlap_here

    return best_overlap

In [ ]:
compute_overlap(image_size=(315,315,49), roi_size=(128,128,32 ), target_num=175)